In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest, RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, ConfusionMatrixDisplay, r2_score
from sklearn.preprocessing import LabelEncoder

### Milk

os.chdir("C:/Python/Datasets")
milk = pd.read_csv("milk.csv", index_col=0)
milk.head()

clf = IsolationForest(contamination=0.05, random_state=25)
clf.fit(milk)
predictions = clf.predict(milk)
predictions

milk.index

# Among the above labels, `-1` indicates suspected outlier. Hence here, *SEAL* and *DOLPHIN* seem to be outliers.

### Nutrient

nut = pd.read_csv("nutrient.csv", index_col=0)

clf = IsolationForest(contamination=0.05, random_state=25)
clf.fit(nut)
predictions = clf.predict(nut)
predictions

df_out = pd.DataFrame({'IDs':list(nut.index), 'label':list(predictions)})
df_out[df_out['label']==-1]

### Glass

glass = pd.read_csv("C:/Python/Cases/Glass Identification/Glass.csv")
X, y = glass.drop('Type', axis=1), glass['Type']

clf = IsolationForest(contamination='auto', random_state=25)
clf.fit(X)
predictions = clf.predict(X)
df_out = pd.DataFrame({'Type':list(y.values), 'label':list(predictions)})
pd.crosstab(df_out['Type'], df_out['label'], margins=True)



# Model with Entire Data

X, y = glass.drop('Type', axis=1), glass['Type']
le = LabelEncoder()
le_y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, le_y,
                                                    test_size=0.3, stratify=y, random_state=25) 

rf = RandomForestClassifier(random_state=25)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_pred_prob = rf.predict_proba(X_test)
print(accuracy_score(y_test, y_pred))
print(log_loss(y_test, y_pred_prob))

cm = confusion_matrix(y_test, y_pred)
plt_cm = ConfusionMatrixDisplay(cm, display_labels=list(le.classes_))
plt_cm.plot()
plt.title("Before Removing Outliers")
plt.show()

# Removing Outliers

glass_in = glass[df_out['label']!=-1]
glass_in.shape

le = LabelEncoder()
glass_in['Type'] = le.fit_transform( glass_in['Type'] )
X, y = glass_in.drop('Type', axis=1), glass_in['Type']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3, stratify=y, random_state=25) 
rf = RandomForestClassifier(random_state=25)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_pred_prob = rf.predict_proba(X_test)
print(accuracy_score(y_test, y_pred))
print(log_loss(y_test, y_pred_prob))

cm = confusion_matrix(y_test, y_pred)
plt_cm = ConfusionMatrixDisplay(cm, display_labels=list(le.classes_))
plt_cm.plot()
plt.title("After Removing Outliers")
plt.show()

# Detect Oultiers from other unlabelled data

tst = pd.read_csv("C:/Python/Cases/Glass Identification/tst_Glass.csv")

clf.predict(tst)

### Concrete Strength

concrete = pd.read_csv("C:/Python/Cases/Concrete Strength/Concrete_Data.csv")
X, y = concrete.drop('Strength', axis=1), concrete['Strength']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                    test_size=0.3, random_state=25) 

rf = RandomForestRegressor(random_state=25)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
r2_score(y_test, y_pred)

# Detecting Outliers

clf = IsolationForest(contamination='auto', random_state=25)
clf.fit(X)
predictions = clf.predict(X)

np.unique(predictions, return_counts=True)

# Removing Outliers

conc_in = concrete[predictions!=-1]
conc_in.shape

X_in, y_in = conc_in.drop('Strength', axis=1), conc_in['Strength']
X_train, X_test, y_train, y_test = train_test_split(X_in, y_in,
                                    test_size=0.3, random_state=25) 
rf = RandomForestRegressor(random_state=25)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
r2_score(y_test, y_pred)

conc_out = concrete[predictions==-1]
conc_out.shape

X_out, y_out = conc_out.drop('Strength', axis=1), conc_out['Strength']
X_train, X_test, y_train, y_test = train_test_split(X_out, y_out,
                                    test_size=0.3, random_state=25) 
rf = RandomForestRegressor(random_state=25)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
r2_score(y_test, y_pred)



tst = pd.read_csv("C:/Python/Cases/Concrete Strength/testConcrete.csv")
clf.fit(X)
clf.predict(tst)